# K-means segmentation
Loads the features built by `build_features.py` (`output/obs_features.parquet`).

1. Choose `K` from the elbow, silhouette and Davies-Bouldin plots (Capitec: 3-5 sensible, 6-8 upper limit).
2. Check the clusters are stable when the data is resampled.
3. Label clusters from the heatmap; validate with bad rates and stability over months.

## Settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

pd.set_option('display.max_columns', None)

FEATURES_FILE = 'output/obs_features.parquet'   # created by build_features.py
RANDOM_STATE = 42

CLUSTER_FEATURES = [
    'avg_period_income',          # income level
    'income_cv',                  # income volatility
    'recent_vs_avg_income',       # income direction (recent vs usual)
    'pay_gap_cv',                 # pay day regularity
    'avg_spend_ratio',            # spend vs pay
    'avg_spend_per_day',          # spending level
    'avg_frac_to_half',           # how quickly pay is spent
    'avg_first3_share',           # spending straight after pay day
    'bal7_ratio_avg',             # share of pay left a week after pay day (usual cycles)
    'bal7_ratio_change',          # latest cycle vs usual (<0 = less left than usual)
    'committed_share_after_pay',  # share of first-3-day outflows that are debit orders
]
LOG_FEATURES = ['avg_period_income', 'avg_spend_per_day']

# 0/1 flags from build_features.py saying WHY a feature was missing (the feature itself was filled)
FLAG_FEATURES = ['no_income_detected', 'few_pay_events', 'no_complete_cycle', 'no_bal7_history']
FLAG_WEIGHT = 1.0    # 0 = ignore flags, lower (e.g. 0.5) if clusters just split on the flags

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

K = 4   # set after looking at the plots below

## Load features

In [ ]:
obs = pd.read_parquet(FEATURES_FILE)
base = obs[obs['exclude_flag'] == 0].copy()

missing = base[CLUSTER_FEATURES].isna().sum().sum()
assert missing == 0, f'{missing} missing values - rerun build_features.py with FILL_MISSING = True'
display(base[FLAG_FEATURES].mean().round(3).to_frame('share_flagged'))

data = base[['obs_id'] + CLUSTER_FEATURES + FLAG_FEATURES].copy()
print(f'Clustering all {len(data):,} observations')

## Prepare: log, winsorize, scale

In [ ]:
X = data[CLUSTER_FEATURES].copy()
for col in LOG_FEATURES:
    X[col] = np.log1p(X[col].clip(lower=0))
X = X.clip(lower=X.quantile(0.01), upper=X.quantile(0.99), axis=1)   # winsorize 1% / 99%

# Scale the features; flags stay 0/1 (times FLAG_WEIGHT) so scaling doesn't blow them up
X_scaled_df = pd.DataFrame(StandardScaler().fit_transform(X), columns=CLUSTER_FEATURES, index=X.index)
flags = data[FLAG_FEATURES]
X_scaled_df = X_scaled_df.join(flags * FLAG_WEIGHT)
X_scaled = X_scaled_df.to_numpy()

# Spearman correlation of the clustering features (|r| > 0.7 = consider dropping one)
corr = X.corr(method='spearman')
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Spearman correlation of clustering features')
plt.tight_layout()
plt.show()

SIL_SAMPLE = min(10_000, len(X_scaled))

## Choose K
- **Elbow**: look for the bend.
- **Silhouette**: higher is better (0.25+ = some structure, 0.5+ = clear).
- **Davies-Bouldin**: lower is better.

In [ ]:
ks = range(2, 9)
rows = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
    rows.append({'k': k, 'inertia': km.inertia_,
                 'silhouette': silhouette_score(X_scaled, km.labels_, sample_size=SIL_SAMPLE,
                                                random_state=RANDOM_STATE),
                 'davies_bouldin': davies_bouldin_score(X_scaled, km.labels_)})
scores = pd.DataFrame(rows).set_index('k')
display(scores.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(ks, scores['inertia'], marker='o')
axes[0].set(title='Elbow (inertia)', xlabel='k')
axes[1].plot(ks, scores['silhouette'], marker='o', color='tab:orange')
axes[1].set(title='Silhouette (higher = better)', xlabel='k')
axes[2].plot(ks, scores['davies_bouldin'], marker='o', color='tab:green')
axes[2].set(title='Davies-Bouldin (lower = better)', xlabel='k')
plt.tight_layout()
plt.show()

## Stability
Refit on bootstrap samples and compare labels with the full fit (adjusted Rand index: 1 = identical, above ~0.8 = stable).

In [ ]:
def stability(M, k, n_boot=10):
    rng = np.random.default_rng(RANDOM_STATE)
    full = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(M)
    scores = []
    for b in range(n_boot):
        idx = rng.choice(len(M), len(M), replace=True)
        km_b = KMeans(n_clusters=k, n_init=10, random_state=b).fit(M[idx])
        scores.append(adjusted_rand_score(full.labels_, km_b.predict(M)))
    return np.mean(scores)

display(pd.Series({k: stability(X_scaled, k) for k in range(3, 7)}, name='mean ARI').round(3).to_frame())

## Fit final model

In [ ]:
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
data['cluster'] = kmeans.labels_
print('Silhouette:', round(silhouette_score(X_scaled, data['cluster'], sample_size=SIL_SAMPLE,
                                            random_state=RANDOM_STATE), 3))

## Cluster profile
Heatmap: mean z-score per cluster (red = above average, blue = below average), clusters ordered by bad rate.

In [ ]:
data['target_value'] = base['target_value'].to_numpy()

profile = data.groupby('cluster').agg(n_obs=('obs_id', 'size'), bad_rate=('target_value', 'mean'))
profile['share_%'] = (profile['n_obs'] / profile['n_obs'].sum() * 100).round(1)
profile['bad_rate_%'] = (profile['bad_rate'] * 100).round(1)
order = profile.sort_values('bad_rate').index
display(profile.loc[order, ['n_obs', 'share_%', 'bad_rate_%']])

display(data.groupby('cluster')[CLUSTER_FEATURES].median().loc[order].round(2))

z = X_scaled_df[CLUSTER_FEATURES].groupby(data['cluster'].to_numpy()).mean().T[order]
lim = min(np.abs(z.values).max(), 2)
fig, ax = plt.subplots(figsize=(1.4 * len(order) + 5, 0.45 * len(z) + 2))
sns.heatmap(z.clip(-lim, lim), annot=z.round(2), fmt='', cmap='RdBu_r', center=0, vmin=-lim, vmax=lim,
            cbar_kws={'label': 'Std devs from average'}, ax=ax)
ax.set_xticklabels([f'C{c}\nbad {profile.loc[c, "bad_rate"]:.0%}\nn={profile.loc[c, "n_obs"]}' for c in order],
                   rotation=0)
ax.set_xlabel('')
ax.set_title('Cluster profiles: red = above average, blue = below average')
plt.tight_layout()
plt.show()

# Share of each cluster with each missing-data flag (filled values sit at the median, so check this)
display(flags.groupby(data['cluster'].to_numpy()).mean().loc[order].round(2).rename_axis('cluster'))

## Bad rate (incl. observations that could not be clustered) and stability over months

In [ ]:
val = base[['obs_id', 'target_value', 'm0_monthenddate', 'product_group', 'm0_start_cd_group']].merge(
    data[['obs_id', 'cluster']], on='obs_id', how='left')
val['cluster'] = val['cluster'].astype('Int64').astype(str).replace('<NA>', 'not clustered')
print(f"Overall bad rate: {val['target_value'].mean() * 100:.1f}%")
display(val.groupby('cluster')['target_value'].agg(n_obs='size', bad_rate='mean')
        .assign(bad_rate=lambda d: (d['bad_rate'] * 100).round(1)))

val['m0_month'] = pd.to_datetime(val['m0_monthenddate']).dt.to_period('M')
print('Cluster share by m0 month (%) - rows should look similar if the clusters are stable')
display(pd.crosstab(val['m0_month'], val['cluster'], normalize='index').mul(100).round(1))
print('Bad rate by m0 month (%)')
display(val.pivot_table(index='m0_month', columns='cluster', values='target_value', aggfunc='mean')
        .mul(100).round(1))

## Static attributes per cluster
How the behavioural clusters relate to the existing static segments.

In [ ]:
display(pd.crosstab(val['cluster'], val['product_group'], normalize='index').mul(100).round(1))
display(pd.crosstab(val['cluster'], val['m0_start_cd_group'], normalize='index').mul(100).round(1))

## 2D view (PCA)

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
idx = np.random.default_rng(RANDOM_STATE).choice(len(coords), min(5_000, len(coords)), replace=False)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=coords[idx, 0], y=coords[idx, 1], hue=data['cluster'].to_numpy()[idx],
                palette='tab10', s=10, alpha=0.6)
plt.title(f'Clusters in 2D (PCA explains {pca.explained_variance_ratio_.sum() * 100:.0f}% of variance)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

## Save labels

In [ ]:
data[['obs_id', 'cluster']].to_parquet('output/clusters_kmeans.parquet', index=False)
print('Saved output/clusters_kmeans.parquet')